# Day 53 — MLOps intro: experiment tracking with MLflow
Objectives:
- Track params, metrics, and artifacts with MLflow.
- Organize experiments and runs.
- Save and load models from MLflow.
Note: `pip install mlflow` (already in requirements.txt). By default this uses a local `mlruns` folder.

In [ ]:
import mlflow
import mlflow.sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
X,y = load_breast_cancer(return_X_y=True)
Xtr,Xte,ytr,yte = train_test_split(X,y, test_size=0.2, random_state=42, stratify=y)
pipe = Pipeline([('sc', StandardScaler()), ('clf', LogisticRegression(max_iter=1000))])
mlflow.set_experiment('ds-60day-bc-experiment')
with mlflow.start_run(run_name='baseline-logreg'):
    mlflow.log_param('model', 'LogisticRegression')
    mlflow.log_param('scale', True)
    pipe.fit(Xtr,ytr)
    yprob = pipe.predict_proba(Xte)[:,1]
    auc = roc_auc_score(yte, yprob)
    mlflow.log_metric('roc_auc', auc)
    mlflow.sklearn.log_model(pipe, artifact_path='model')
    print('ROC AUC:', auc)


## Viewing results
Run the MLflow UI in a terminal:
```bash
mlflow ui --backend-store-uri mlruns
```
Then open http://127.0.0.1:5000 to view experiments.

## Loading a model from MLflow
You can load the saved model artifact and use it for inference.

In [ ]:
# Example: load the last logged model (adjust run_id/artifact URI as needed)
# model_uri = 'runs:/<run_id>/model'
# loaded = mlflow.sklearn.load_model(model_uri)
# loaded.predict(Xte[:5])


## Learner exercises and progressive hints

1. Log additional parameters such as Logistic Regression `C` and compare runs.
2. Save a confusion-matrix PNG and log it as an artifact.
3. Try a different classifier, such as Random Forest, and compare ROC AUC.

### Progressive hints

1. Make one run per configuration and include split seed, metric name, and model
   type. Avoid changing several uncontrolled factors at once.
2. Save figures under an ignored `artifacts/` directory, close the figure, and
   pass the path to `mlflow.log_artifact`.
3. Reuse exactly the same train/test split. Compare runtime and complexity as
   well as score.

The reference solution adds scikit-learn autologging and model reload. Start
with explicit logging so you know which information is essential; use autolog
as a supplement, not as a substitute for experiment design.

### Additional mastery practice

Make experiment records reconstructable: status, parameters, data/code identity, metrics, artifacts, and model signature must describe one coherent run.

Predict or plan before you run code. Use the hint only after an honest
attempt, and record the evidence that would prove your result correct.

4. **Failure-state handling:** Run an experiment that intentionally raises after logging parameters. Verify MLflow records a failed status and useful exception context without exposing raw data or secrets.
   **Progressive hint:** Use the run context manager so exception exit marks the run failed. Log safe stage/status information before re-raising.
5. **Provenance manifest:** Log a JSON provenance artifact containing data fingerprint, code revision, dependency lock hash, feature schema, split policy, and metric definitions.
   **Progressive hint:** Use portable identifiers and hashes, not developer-specific absolute paths. Validate required fields before ending the run.
6. **Reload and signature check:** Log a fitted pipeline with an input example/signature, reload it by run URI, and assert prediction parity on a fixed fixture.
   **Progressive hint:** The fixture must use the documented schema and never come from hidden notebook state. Compare probabilities within a tolerance.

Before opening the reference solution, explain the relevant assumption,
failure mode, and validation check for every answer.


In [ ]:
# Expanded mastery lab scratch space
#
# Keep the official solution closed until you have attempted each task.
# Add small assertions, shape checks, or metric comparisons as evidence.

# Practice 4 — Failure-state handling


# Practice 5 — Provenance manifest


# Practice 6 — Reload and signature check
